In [ ]:
'''
project='ee-chandanagalgali'
'''

"\nproject='ee-chandanagalgali'\n"

In [ ]:
'''
2023-12-10
'''

'\n2023-12-10\n'

In [ ]:
'''
pollutants_info = {
    'NO2': {'collection': 'COPERNICUS/S5P/OFFL/L3_NO2', 'band': 'tropospheric_NO2_column_number_density', 'start_date': '2018-06-28'},
    'CO': {'collection': 'COPERNICUS/S5P/OFFL/L3_CO', 'band': 'CO_column_number_density', 'start_date': '2018-06-28'},
    'CH4': {'collection': 'COPERNICUS/S5P/OFFL/L3_CH4', 'band': 'CH4_column_volume_mixing_ratio_dry_air', 'start_date': '2019-02-08'},
    'O3': {'collection': 'COPERNICUS/S5P/OFFL/L3_O3', 'band': 'O3_column_number_density', 'start_date': '2018-07-10'},
    'SO2': {'collection': 'COPERNICUS/S5P/OFFL/L3_SO2', 'band': 'SO2_column_number_density', 'start_date': '2018-07-10'},
    'HCHO': {'collection': 'COPERNICUS/S5P/OFFL/L3_HCHO', 'band': 'tropospheric_HCHO_column_number_density', 'start_date': '2018-10-02'},
    'AER_AI': {'collection': 'COPERNICUS/S5P/OFFL/L3_AER_AI', 'band': 'absorbing_aerosol_index', 'start_date': '2018-07-04'}
}
'''

"\npollutants_info = {\n    'NO2': {'collection': 'COPERNICUS/S5P/OFFL/L3_NO2', 'band': 'tropospheric_NO2_column_number_density', 'start_date': '2018-06-28'},\n    'CO': {'collection': 'COPERNICUS/S5P/OFFL/L3_CO', 'band': 'CO_column_number_density', 'start_date': '2018-06-28'},\n    'CH4': {'collection': 'COPERNICUS/S5P/OFFL/L3_CH4', 'band': 'CH4_column_volume_mixing_ratio_dry_air', 'start_date': '2019-02-08'},\n    'O3': {'collection': 'COPERNICUS/S5P/OFFL/L3_O3', 'band': 'O3_column_number_density', 'start_date': '2018-07-10'},\n    'SO2': {'collection': 'COPERNICUS/S5P/OFFL/L3_SO2', 'band': 'SO2_column_number_density', 'start_date': '2018-07-10'},\n    'HCHO': {'collection': 'COPERNICUS/S5P/OFFL/L3_HCHO', 'band': 'tropospheric_HCHO_column_number_density', 'start_date': '2018-10-02'},\n    'AER_AI': {'collection': 'COPERNICUS/S5P/OFFL/L3_AER_AI', 'band': 'absorbing_aerosol_index', 'start_date': '2018-07-04'}\n}\n"

In [14]:
import ee
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from datetime import datetime
from sklearn.model_selection import train_test_split

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Define Mumbai shapefile
mumbai = ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file")

# Define NO2 information
NO2_INFO = {
    'collection': 'COPERNICUS/S5P/OFFL/L3_NO2',
    'band': 'tropospheric_NO2_column_number_density',
    'start_date': '2018-06-28'
}

# Fetch Sentinel-2 image
def fetch_sentinel2_image(start_date, end_date):
    collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(start_date, end_date) \
        .filterBounds(mumbai) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    image = collection.median().select(["B2", "B3", "B4", "B8"]).clip(mumbai)
    return image

# Fetch NO2 image
def fetch_no2_image(start_date, end_date):
    collection = ee.ImageCollection(NO2_INFO['collection']) \
        .select(NO2_INFO['band']) \
        .filterDate(start_date, end_date)
    image = collection.mean().clip(mumbai)
    return image

# Sample training data
def sample_training_data(start_date, end_date):
    sentinel2_image = fetch_sentinel2_image(start_date, end_date)
    no2_image = fetch_no2_image(start_date, end_date)

    combined_image = sentinel2_image.addBands(no2_image)
    samples = combined_image.sample(
        region=mumbai,
        scale=100,
        numPixels=5000,
        geometries=False
    ).getInfo()

    predictors = []
    target = []

    for feature in samples['features']:
        props = feature['properties']
        if all(band in props for band in ["B2", "B3", "B4", "B8"]) and NO2_INFO['band'] in props:
            predictors.append([props["B2"], props["B3"], props["B4"], props["B8"]])
            target.append(props[NO2_INFO['band']])

    return np.array(predictors), np.array(target)

# Train the model
def train_model(predictors, target):
    scaler = StandardScaler()
    predictors = scaler.fit_transform(predictors)  # Normalize predictors
    target = target.reshape(-1, 1)  # Reshape target
    target = scaler.fit_transform(target)

    predictors = predictors.reshape(-1, 2, 2, 1)  # Reshape for CNN

    # Split into train and validation sets
    x_train, x_val, y_train, y_val = train_test_split(predictors, target, test_size=0.2, random_state=42)

    model = build_cnn_model((2, 2, 1))  # Define input shape for CNN
    model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=100, batch_size=32)
    return model, scaler  # Return model and scaler

@tf.keras.utils.register_keras_serializable()
def custom_mse(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

# Build CNN model
def build_cnn_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (2, 2), activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling2D((1, 1)),
        tf.keras.layers.Conv2D(64, (1, 1), activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(1, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

if __name__ == "__main__":
    start_date = '2018-06-28'
    end_date = datetime.now().strftime('%Y-%m-%d')

    print("Sampling training data.")
    predictors, target = sample_training_data(start_date, end_date)

    print("Training the CNN model.")
    model, scaler = train_model(predictors, target)  # Correct unpacking

    print("Saving model and scaler.")
    model.save('/content/NO2_cnn_model.h5')  # Save the trained model
    np.save('/content/predictor_scaler_mean.npy', scaler.mean_)  # Save the scaler mean
    np.save('/content/predictor_scaler_scale.npy', scaler.scale_)  # Save the scaler scale

Sampling training data.
Training the CNN model.
Epoch 1/100


/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


123/123 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.9511 - mae: 0.8071 - val_loss: 0.9050 - val_mae: 0.7749
Epoch 2/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8936 - mae: 0.7764 - val_loss: 0.8692 - val_mae: 0.7628
Epoch 3/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.8619 - mae: 0.7618 - val_loss: 0.8562 - val_mae: 0.7587
Epoch 4/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.8010 - mae: 0.7348 - val_loss: 0.8511 - val_mae: 0.7627
Epoch 5/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.8101 - mae: 0.7410 - val_loss: 0.8430 - val_mae: 0.7559
Epoch 6/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.8222 - mae: 0.7459 - val_loss: 0.8444 - val_mae: 0.7557
Epoch 7/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.8100 - mae: 0.7408 - val_loss: 0.8317 - val_mae: 0.7444
Epoch 8/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.8172 - mae: 0.7441 - val_loss: 0.8311 - val_mae: 0.7429
Epoch 9/100
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss:

Saving model and scaler.


In [18]:
import ee
import tensorflow as tf
import numpy as np

# Initialize Earth Engine
try:
    ee.Initialize(project='ee-chandanagalgali')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='ee-chandanagalgali')

# Register the custom loss function
@tf.keras.utils.register_keras_serializable()
def custom_mse(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

# Load trained model and scalers
model = tf.keras.models.load_model('/content/NO2_cnn_model.h5', custom_objects={'custom_mse': custom_mse, 'mse': tf.keras.losses.MeanSquaredError()})  # Add 'mse' to custom_objects
scaler_mean = np.load('/content/predictor_scaler_mean.npy')
scaler_scale = np.load('/content/predictor_scaler_scale.npy')

# Use the exact Mumbai shapefile geometry
def get_mumbai_geometry():
    print("Using the exact Mumbai shapefile outline.")
    return ee.FeatureCollection("projects/ee-chandanagalgali/assets/mumbai_ward_shape_file").geometry()

mumbai_geometry = get_mumbai_geometry()

# Fetch image with fallbacks
def fetch_image(user_date):
    start_date = ee.Date(user_date)
    end_date = start_date.advance(1, 'day')

    # Sentinel-2
    s2_image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
        .select(['B2', 'B3', 'B4', 'B8'])
    if s2_image.size().getInfo() > 0:
        print("Using Sentinel-2 data.")
        return s2_image.median()

    # MODIS Fallback
    print("No Sentinel-2 data. Using MODIS data.")
    modis_image = ee.ImageCollection("MODIS/061/MCD43A4") \
        .filterBounds(mumbai_geometry) \
        .filterDate(start_date, end_date) \
        .select(['Nadir_Reflectance_Band1', 'Nadir_Reflectance_Band4', 'Nadir_Reflectance_Band3', 'Nadir_Reflectance_Band2']) \
        .median().rename(['B2', 'B3', 'B4', 'B8'])
    return modis_image

# Divide region into smaller tiles
def generate_tiles(geometry, num_tiles=4):
    print("Dividing region into smaller tiles...")
    bounds = geometry.bounds().coordinates().get(0).getInfo()
    lon_min, lat_min = bounds[0]
    lon_max, lat_max = bounds[2]

    lon_step = (lon_max - lon_min) / num_tiles
    lat_step = (lat_max - lat_min) / num_tiles

    tiles = []
    for i in range(num_tiles):
        for j in range(num_tiles):
            tile = ee.Geometry.Rectangle([
                lon_min + i * lon_step,
                lat_min + j * lat_step,
                lon_min + (i + 1) * lon_step,
                lat_min + (j + 1) * lat_step
            ])
            tiles.append(tile)
    return tiles

# Process a tile and make predictions
def process_tile(tile, image):
    try:
        sampled_data = image.sample(
            region=tile,
            scale=100,  # Resolution for sampling
            geometries=False,
            numPixels=5000
        ).getInfo()

        inputs = []
        for feature in sampled_data['features']:
            props = feature['properties']
            inputs.append([props['B2'], props['B3'], props['B4'], props['B8']])

        inputs = np.array(inputs)
        if inputs.size == 0:
            return None, None

        # Normalize inputs
        inputs = (inputs - scaler_mean) / scaler_scale

        # Reshape for CNN model
        inputs = inputs.reshape(-1, 2, 2, 1)  # Match CNN input shape
        predictions = model.predict(inputs).flatten()

        # Take mean prediction for this tile
        mean_prediction = np.clip(predictions.mean(), 0, 0.0005)
        return mean_prediction, tile
    except Exception as e:
        print(f"Error processing tile: {e}")
        return None, None

# Predict and export NO2 layer
def predict_no2(user_date):
    try:
        print(f"Processing NO2 for {user_date}.")
        image = fetch_image(user_date)
        if image is None:
            raise ValueError("No valid image data available.")

        print(f"Bands in the image: {image.bandNames().getInfo()}")
        tiles = generate_tiles(mumbai_geometry, num_tiles=4)
        predictions = []

        for idx, tile in enumerate(tiles):
            print(f"Processing tile {idx + 1} of {len(tiles)}...")
            mean_prediction, processed_tile = process_tile(tile, image)
            if mean_prediction is not None:
                predictions.append(ee.Image.constant(mean_prediction).clip(processed_tile).rename('Downscaled_NO2'))

        if not predictions:
            print("No valid predictions generated.")
            return

        # Combine tiles into final image
        print("Combining predictions into final image...")
        downscaled_no2 = ee.ImageCollection(predictions).mosaic().clip(mumbai_geometry)

        # Export Downscaled NO2 Layer
        print("Exporting Downscaled_NO2 layer...")
        task = ee.batch.Export.image.toAsset(
            image=downscaled_no2,
            description='Downscaled_NO2',
            assetId='projects/ee-chandanagalgali/assets/downscaled_NO2_final',
            region=mumbai_geometry,
            scale=100,  # Export at 100m resolution
            maxPixels=1e13
        )
        task.start()
        print("Export task started. Monitor progress in GEE.")

    except Exception as e:
        print(f"Error processing NO2 pixel-wise: {e}")
        print("Failed to create downscaled NO₂ map.")

if __name__ == "__main__":
    user_date = input("Enter the date for prediction (YYYY-MM-DD): ")
    predict_no2(user_date)
    print("Processing complete.")

Using the exact Mumbai shapefile outline.
Enter the date for prediction (YYYY-MM-DD): 2023-12-10
Processing NO2 for 2023-12-10.
No Sentinel-2 data. Using MODIS data.
Bands in the image: ['B2', 'B3', 'B4', 'B8']
Dividing region into smaller tiles...
Processing tile 1 of 16...
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Processing tile 2 of 16...
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Processing tile 3 of 16...
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Processing tile 4 of 16...
155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Processing tile 5 of 16...
146/146 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Processing tile 6 of 16...
149/149 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step
Processing tile 7 of 16...
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
Processing tile 8 of 16...
153/153 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Processing tile 9 of 16...
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Processing tile 10 of 16...
147/147 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Processing tile 11 of 16...
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Proce